# Optimizing using Optune on Imbalanced dataset Multi-task

In [ ]:
import os
import math
import json
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.optim import AdamW
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
)
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold
from gensim.models import KeyedVectors

# ================== TRANSFORMERS (LaBSE) ==================
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup

import optuna
from optuna.trial import TrialState
import matplotlib.pyplot as plt

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

# ================== CONFIG ==================
CHECKPOINT_DIR = "outputModel/LaBSE-Optimizing"
BEST_NAME_MODEL = "best_model_multi_task_emoji_attention_fold"
TRAIN_CSV_PATH = "dataset/4. newlabel/extend/train.csv"
TEST_CSV_PATH  = "dataset/4. newlabel/extend/test.csv"
OUTPUT_PRED_BEST = "outputPrediksi/LaBSE-Optimizing/pred_(EmojiAttention).csv"
OUTPUT_THRESHOLDS = "outputPrediksi/LaBSE-Optimizing/opt_thresholdsFeedEmojiAttn.json"       # file 1
OUTPUT_TEMPERATURE = "outputPrediksi/LaBSE-Optimizing/opt_temperatureFeedEmojiAttn.json"     # file 2 (baru)

MAX_LEN    = 128
BATCH_SIZE = 8
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DROPOUT    = 0.05
LR         = 1e-6

N_TRIALS   = 10  # <<< should be 20
NUM_EPOCHS = 5  # <<< should be 10
KFOLD_SPLITS = 1  # <<< FIX: define k-fold splits

# Train extras
WEIGHT_DECAY   = 0.0
WARMUP_RATIO   = 0.0
GRAD_ACCUM     = 1
MAX_GRAD_NORM  = 0.5

# Loss mixing across tasks
LOSS_W_EMO = 1.0
LOSS_W_SEN = 1.0
LOSS_W_ASP = 1.0

# Focal Loss toggle
FOCAL_GAMMA_EMO = 0.0
FOCAL_GAMMA_SEN = 0.0
FOCAL_GAMMA_ASP = 0.0
FOCAL_ALPHA_EMO = 0.25
FOCAL_ALPHA_SEN = None
FOCAL_ALPHA_ASP = None

# Default threshold fallback
DEFAULT_THRESHOLD = 0.5

EMOTION_LABELS = ['anger','anticipation','disgust','fear','joy','sadness','surprise','trust']
NUM_EMOTIONS  = len(EMOTION_LABELS)
NUM_SENTIMENT = 3
NUM_ASPECT    = 5

# ================== LOAD EMOJI2VEC ==================
print("Loading emoji2vec model...")
emoji2vec = KeyedVectors.load_word2vec_format("emoji2vec.bin", binary=True)
EMOJI_DIM = emoji2vec.vector_size

# ================== HELPERS ==================
def class_weights_from_counts(counts, num_classes):
    counts = np.clip(counts.astype(float), 1.0, None)
    inv = 1.0 / counts
    inv = inv * (num_classes / inv.sum())
    return torch.tensor(inv, dtype=torch.float)

def pos_weight_from_binary_counts(pos_counts, total):
    pos = np.clip(pos_counts.astype(float), 1.0, None)
    neg = np.maximum(total - pos, 1.0)
    pw = neg / pos
    return torch.tensor(pw, dtype=torch.float)

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def softmax_np(x, axis=1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def tune_thresholds_per_label(y_true, y_prob):
    fine = np.linspace(0.05, 0.95, 37)
    extra = np.array([0.01, 0.02, 0.03, 0.97, 0.98, 0.99])
    grid = np.unique(np.concatenate([fine, extra]))
    thresholds = []
    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_prob[:, j]
        if len(np.unique(yt)) < 2:
            thresholds.append(0.5); continue
        best_f1, best_t = -1.0, 0.5
        for t in grid:
            pred = (yp >= t).astype(int)
            f1 = f1_score(yt, pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        thresholds.append(best_t)
    return thresholds

def tune_temperature_for_multilabel(y_true, val_logits, temps=np.linspace(0.7, 1.5, 17)):
    best_T, best_f1 = 1.0, -1.0
    for T in temps:
        probs = 1.0 / (1.0 + np.exp(-(val_logits / T)))
        macro_f1 = f1_score(y_true, (probs >= 0.5).astype(int), average='macro', zero_division=0)
        if macro_f1 > best_f1:
            best_f1, best_T = macro_f1, float(T)
    return best_T

# ================== FOCAL LOSS (optional) ==================
class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, pos_weight=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.pos_weight = pos_weight
        self.reduction = reduction
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none', pos_weight=self.pos_weight
        )
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha_t * ((1 - pt) ** self.gamma) * bce
        if self.reduction == "mean": return focal.mean()
        elif self.reduction == "sum": return focal.sum()
        return focal

class FocalCrossEntropy(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
    def forward(self, logits, targets):
        log_prob = nn.functional.log_softmax(logits, dim=1)
        prob = torch.exp(log_prob)
        ce = nn.functional.nll_loss(log_prob, targets, weight=self.weight, reduction='none')
        pt = prob[torch.arange(prob.size(0), device=prob.device), targets]
        focal = ((1 - pt) ** self.gamma) * ce
        if self.reduction == "mean": return focal.mean()
        elif self.reduction == "sum": return focal.sum()
        return focal

# ================== MODEL ==================
class EmojiAttention(nn.Module):
    def __init__(self, emoji_dim, hidden_dim):
        super().__init__()
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key   = nn.Linear(emoji_dim, hidden_dim)
        self.value = nn.Linear(emoji_dim, hidden_dim)
        self.scale = math.sqrt(hidden_dim)
    def forward(self, text_feat, emoji_feat):
        Q = self.query(text_feat)
        K = self.key(emoji_feat)
        V = self.value(emoji_feat)
        attn_score  = torch.sum(Q * K, dim=-1, keepdim=True) / self.scale
        attn_weight = torch.sigmoid(attn_score)
        attended_emoji = attn_weight * V
        return attended_emoji

class MultiTaskWithEmojiAttention(nn.Module):
    def __init__(self, num_emotions=NUM_EMOTIONS, num_sentiment=NUM_SENTIMENT, num_aspect=NUM_ASPECT,
                 dropout=DROPOUT, emoji_dim=EMOJI_DIM, pooling: str = "mean"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("sentence-transformers/LaBSE")
        self.dropout = nn.Dropout(dropout)
        self.pooling = pooling
        hidden_size = self.backbone.config.hidden_size

        self.emoji_attn = EmojiAttention(emoji_dim, hidden_size)
        self.emoji_fc   = nn.Linear(emoji_dim, hidden_size)

        # Gating parameters (0..1 via sigmoid)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))

        self.classifier_emotion   = nn.Linear(hidden_size, num_emotions)
        self.classifier_sentiment = nn.Linear(hidden_size, num_sentiment)
        self.classifier_aspect    = nn.Linear(hidden_size, num_aspect)

    def forward(self, input_ids, attention_mask, emoji_vec):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        if self.pooling == "mean":
            pooled = mean_pooling(outputs.last_hidden_state, attention_mask)
        else:
            pooled = outputs.last_hidden_state[:, 0]

        emoji_proj     = self.emoji_fc(emoji_vec)
        emoji_attended = self.emoji_attn(pooled, emoji_vec)

        alpha = torch.sigmoid(self.alpha)
        beta  = torch.sigmoid(self.beta)
        fused = pooled + alpha * emoji_proj + beta * emoji_attended

        x = self.dropout(fused)
        logits_emotion   = self.classifier_emotion(x)
        logits_sentiment = self.classifier_sentiment(x)
        logits_aspect    = self.classifier_aspect(x)
        return logits_emotion, logits_sentiment, logits_aspect

# ================== DATASET ==================
def _safe_parse_emoji_string(s):
    if not isinstance(s, str) or s.strip() == "":
        return []
    s = s.strip()
    if (s.startswith('[') and s.endswith(']')) or ('","' in s):
        try:
            arr = json.loads(s.replace("'", '"'))
            if isinstance(arr, list):
                return [e for e in arr if isinstance(e, str) and len(e) > 0]
        except Exception:
            pass
    return [ch for ch in s]

class MultiTaskDatasetWithEmoji(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['feed_no_emo'].astype(str).tolist()
        self.emojis = df['_emo'].fillna("").tolist()
        self.emotions = df[EMOTION_LABELS].values.astype(float)
        self.sentiments = df['sentiment'].astype(int).values
        self.aspects = df['aspect'].astype(int).values
        self.tokenizer = tokenizer
        self.max_len = max_len
    def emoji_to_vec(self, emoji_str):
        emolist = _safe_parse_emoji_string(emoji_str)
        if len(emolist) == 0:
            return np.zeros(EMOJI_DIM, dtype=np.float32)
        vecs = []
        for e in emolist:
            if e in emoji2vec:
                vecs.append(emoji2vec[e])
        if not vecs:
            return np.zeros(EMOJI_DIM, dtype=np.float32)
        return np.mean(vecs, axis=0).astype(np.float32)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        emoji_vec = self.emoji_to_vec(self.emojis[idx])
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'emoji_vec': torch.tensor(emoji_vec, dtype=torch.float),
            'emotion_labels': torch.tensor(self.emotions[idx], dtype=torch.float),
            'sentiment_label': torch.tensor(self.sentiments[idx], dtype=torch.long),
            'aspect_label': torch.tensor(self.aspects[idx], dtype=torch.long)
        }

# ================== METRICS ==================
def evaluate_multilabel(y_true, y_prob, threshold=0.5, thresholds_per_label=None):
    if thresholds_per_label is None:
        y_pred = (y_prob >= threshold).astype(int)
    else:
        thr = np.array(thresholds_per_label)[None, :]
        y_pred = (y_prob >= thr).astype(int)
    subset_acc = accuracy_score(y_true, y_pred)
    sample_acc = np.mean((y_true == y_pred).sum(axis=1) / y_true.shape[1])
    micro_prec = precision_score(y_true, y_pred, average='micro', zero_division=0)
    micro_rec  = recall_score(y_true, y_pred, average='micro', zero_division=0)
    micro_f1   = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    per_label_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_label_rec  = recall_score(y_true, y_pred, average=None, zero_division=0)
    per_label_f1   = f1_score(y_true, y_pred, average=None, zero_division=0)
    per_label_auc = []
    for j in range(y_true.shape[1]):
        y_col = y_true[:, j]
        try:
            auc = roc_auc_score(y_col, y_prob[:, j]) if len(np.unique(y_col)) == 2 else np.nan
        except ValueError:
            auc = np.nan
        per_label_auc.append(auc)
    valid_aucs = [a for a in per_label_auc if not (a is None or np.isnan(a))]
    macro_auc = np.mean(valid_aucs) if len(valid_aucs) > 0 else np.nan
    return {
        "subset_accuracy": subset_acc,
        "sample_accuracy": sample_acc,
        "micro_precision": micro_prec,
        "micro_recall": micro_rec,
        "micro_f1": micro_f1,
        "macro_precision": macro_prec,
        "macro_recall": macro_rec,
        "macro_f1": macro_f1,
        "macro_auc": macro_auc,
        "per_label_precision": per_label_prec,
        "per_label_recall": per_label_rec,
        "per_label_f1": per_label_f1,
        "per_label_auc": per_label_auc
    }

def print_metrics(metrics, label_names):
    print("\n===== EMOTION EVALUATION METRICS =====")
    print(f"Subset Accuracy   : {metrics['subset_accuracy']:.4f}")
    print(f"Sample Accuracy   : {metrics['sample_accuracy']:.4f}")
    print(f"Micro Precision   : {metrics['micro_precision']:.4f}")
    print(f"Micro Recall      : {metrics['micro_recall']:.4f}")
    print(f"Micro F1          : {metrics['micro_f1']:.4f}")
    print(f"Macro Precision   : {metrics['macro_precision']:.4f}")
    print(f"Macro Recall      : {metrics['macro_recall']:.4f}")
    print(f"Macro F1          : {metrics['macro_f1']:.4f}")
    print(f"Macro ROC-AUC     : {metrics['macro_auc']:.4f}")
    print("\nPer-Label Metrics:")
    for i, name in enumerate(label_names):
        auc_val = metrics['per_label_auc'][i]
        auc_str = f"{auc_val:.4f}" if (auc_val == auc_val) else "NaN"
        print(f"- {name:22s} | P: {metrics['per_label_precision'][i]:.4f} "
              f"| R: {metrics['per_label_recall'][i]:.4f} "
              f"| F1: {metrics['per_label_f1'][i]:.4f} "
              f"| AUC: {auc_str}")

# ================== TRAIN / VAL ==================
def make_losses(emotion_pos_weight, sent_weights, asp_weights):
    if FOCAL_GAMMA_EMO > 0:
        loss_fn_emotion = FocalBCEWithLogitsLoss(
            gamma=FOCAL_GAMMA_EMO, alpha=FOCAL_ALPHA_EMO, pos_weight=emotion_pos_weight.to(DEVICE)
        )
    else:
        loss_fn_emotion = nn.BCEWithLogitsLoss(pos_weight=emotion_pos_weight.to(DEVICE))
    if FOCAL_GAMMA_SEN > 0:
        loss_fn_sentiment = FocalCrossEntropy(gamma=FOCAL_GAMMA_SEN, weight=sent_weights.to(DEVICE))
    else:
        loss_fn_sentiment = nn.CrossEntropyLoss(weight=sent_weights.to(DEVICE))
    if FOCAL_GAMMA_ASP > 0:
        loss_fn_aspect = FocalCrossEntropy(gamma=FOCAL_GAMMA_ASP, weight=asp_weights.to(DEVICE))
    else:
        loss_fn_aspect = nn.CrossEntropyLoss(weight=asp_weights.to(DEVICE))
    return loss_fn_emotion, loss_fn_sentiment, loss_fn_aspect

# <<< FIX: robust function (accept Dataset OR Subset)
def collect_counts_from_subset(subset_or_dataset):
    # If Subset, use its indices; otherwise, use all indices of Dataset
    if isinstance(subset_or_dataset, Subset):
        ds = subset_or_dataset.dataset
        idxs = subset_or_dataset.indices
    else:
        ds = subset_or_dataset
        idxs = np.arange(len(ds))

    if isinstance(idxs, np.ndarray):
        emo_mat = ds.emotions[idxs, :]
        sen_arr = ds.sentiments[idxs]
        asp_arr = ds.aspects[idxs]
    else:
        emo_mat = np.array([ds.emotions[i] for i in idxs])
        sen_arr = np.array([ds.sentiments[i] for i in idxs])
        asp_arr = np.array([ds.aspects[i] for i in idxs])

    emo_pos_counts = emo_mat.sum(axis=0)
    total_samples = emo_mat.shape[0]
    sen_counts = np.bincount(sen_arr, minlength=NUM_SENTIMENT)
    asp_counts = np.bincount(asp_arr, minlength=NUM_ASPECT)
    return emo_pos_counts, total_samples, sen_counts, asp_counts

def train_model(model, train_loader, val_loader, device, epochs, lr, loss_fns,
                weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
                grad_accum_steps=GRAD_ACCUM, max_grad_norm=MAX_GRAD_NORM):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = (len(train_loader) * epochs) // max(1, grad_accum_steps)
    warmup_steps = max(1, int(warmup_ratio * total_steps))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    loss_fn_emotion, loss_fn_sentiment, loss_fn_aspect = loss_fns
    best_val_loss = float('inf')
    best_state, best_val_cache = None, None
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        running = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f"Training {epoch}", leave=False)):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            emoji_vec = batch['emoji_vec'].to(device)
            emotion_labels = batch['emotion_labels'].to(device)
            sentiment_label = batch['sentiment_label'].to(device)
            aspect_label = batch['aspect_label'].to(device)

            logits_emotion, logits_sentiment, logits_aspect = model(input_ids, attention_mask, emoji_vec)
            loss_emotion   = loss_fn_emotion(logits_emotion, emotion_labels)
            loss_sentiment = loss_fn_sentiment(logits_sentiment, sentiment_label)
            loss_aspect    = loss_fn_aspect(logits_aspect, aspect_label)
            loss = LOSS_W_EMO * loss_emotion + LOSS_W_SEN * loss_sentiment + LOSS_W_ASP * loss_aspect

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            # <<< FIX: step on last partial accumulation too
            if ((step + 1) % grad_accum_steps == 0) or (step + 1 == len(train_loader)):
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            running += loss.item()

        val_loss, val_cache = validate_one_epoch(model, val_loader, device, loss_fns)
        print(f"Epoch {epoch:02d} | Train Loss: {running / max(1, len(train_loader)):.4f} | Val Loss: {val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best_val_cache = val_cache
    return best_state, best_val_loss, best_val_cache

@torch.no_grad()
def validate_one_epoch(model, val_loader, device, loss_fns):
    loss_fn_emotion, loss_fn_sentiment, loss_fn_aspect = loss_fns
    model.eval()
    total_val_loss = 0.0
    all_logits_emotion = []; all_probs_emotion = []; all_true_emotion = []
    all_true_sen = []; all_true_asp = []; all_logits_sen = []; all_logits_asp = []
    for batch in tqdm(val_loader, desc="Validating", leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        emoji_vec = batch['emoji_vec'].to(device)
        emotion_labels = batch['emotion_labels'].to(device)
        sentiment_label = batch['sentiment_label'].to(device)
        aspect_label = batch['aspect_label'].to(device)
        logits_emotion, logits_sentiment, logits_aspect = model(input_ids, attention_mask, emoji_vec)
        loss_emotion   = loss_fn_emotion(logits_emotion, emotion_labels)
        loss_sentiment = loss_fn_sentiment(logits_sentiment, sentiment_label)
        loss_aspect    = loss_fn_aspect(logits_aspect, aspect_label)
        loss = LOSS_W_EMO * loss_emotion + LOSS_W_SEN * loss_sentiment + LOSS_W_ASP * loss_aspect
        total_val_loss += loss.item()
        probs_emotion = torch.sigmoid(logits_emotion)
        all_logits_emotion.append(logits_emotion.cpu().numpy())
        all_probs_emotion.append(probs_emotion.cpu().numpy())
        all_true_emotion.append(emotion_labels.cpu().numpy())
        all_true_sen.append(sentiment_label.cpu().numpy())
        all_true_asp.append(aspect_label.cpu().numpy())
        all_logits_sen.append(logits_sentiment.cpu().numpy())
        all_logits_asp.append(logits_aspect.cpu().numpy())
    avg_val_loss = total_val_loss / max(1, len(val_loader))
    all_logits_emotion = np.vstack(all_logits_emotion) if all_logits_emotion else np.empty((0, NUM_EMOTIONS))
    all_probs_emotion  = np.vstack(all_probs_emotion)  if all_probs_emotion  else np.empty((0, NUM_EMOTIONS))
    all_true_emotion   = np.vstack(all_true_emotion)   if all_true_emotion   else np.empty((0, NUM_EMOTIONS))
    all_true_sen = np.concatenate(all_true_sen) if all_true_sen else np.empty((0,))
    all_true_asp = np.concatenate(all_true_asp) if all_true_asp else np.empty((0,))
    all_logits_sen = np.vstack(all_logits_sen) if all_logits_sen else np.empty((0, NUM_SENTIMENT))
    all_logits_asp = np.vstack(all_logits_asp) if all_logits_asp else np.empty((0, NUM_ASPECT))
    return avg_val_loss, (all_logits_emotion, all_probs_emotion, all_true_emotion,
                          all_logits_sen, all_true_sen, all_logits_asp, all_true_asp)

# === Objective function ===
def objective(trial):
    # Suggested hyperparameters
    max_len = trial.suggest_categorical("max_len", [128, 192, 256])
    batch_sz = trial.suggest_categorical("batch_size", [8, 16, 24, 32])
    dropout = trial.suggest_float("dropout", 0.05, 0.5, step=0.05)
    lr = trial.suggest_float("lr", 1e-6, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    grad_accum = trial.suggest_int("grad_accum", 1, 4)
    max_grad_norm = trial.suggest_float("max_grad_norm", 0.5, 5.0)
    loss_w_emo = trial.suggest_float("loss_w_emo", 0.5, 2.0)
    loss_w_sen = trial.suggest_float("loss_w_sen", 0.5, 2.0)
    loss_w_asp = trial.suggest_float("loss_w_asp", 0.5, 2.0)
    focal_gamma_emo = trial.suggest_categorical("focal_gamma_emo", [0.0, 1.0, 2.0, 3.0])
    focal_alpha_emo = trial.suggest_float("focal_alpha_emo", 0.1, 0.75) if focal_gamma_emo > 0 else 0.25

    # Override global constants
    global MAX_LEN, BATCH_SIZE, DROPOUT, LR, WEIGHT_DECAY, WARMUP_RATIO
    global GRAD_ACCUM, MAX_GRAD_NORM
    global LOSS_W_EMO, LOSS_W_SEN, LOSS_W_ASP
    global FOCAL_GAMMA_EMO, FOCAL_ALPHA_EMO

    MAX_LEN = max_len
    BATCH_SIZE = batch_sz
    DROPOUT = dropout
    LR = lr
    WEIGHT_DECAY = weight_decay
    WARMUP_RATIO = warmup_ratio
    GRAD_ACCUM = grad_accum
    MAX_GRAD_NORM = max_grad_norm
    LOSS_W_EMO = loss_w_emo
    LOSS_W_SEN = loss_w_sen
    LOSS_W_ASP = loss_w_asp
    FOCAL_GAMMA_EMO = focal_gamma_emo
    FOCAL_ALPHA_EMO = focal_alpha_emo

    # Reload dataset once
    df = pd.read_csv(TRAIN_CSV_PATH)
    df_train = df.sample(frac=0.85, random_state=42)
    df_val = df.drop(df_train.index)

    dataset_train = MultiTaskDatasetWithEmoji(df_train, tokenizer, MAX_LEN)
    dataset_val = MultiTaskDatasetWithEmoji(df_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)

    # Compute weights
    emo_pos_counts, total_samples, sen_counts, asp_counts = collect_counts_from_subset(dataset_train)  # <<< FIX: now valid for Dataset
    emo_pos_weight = pos_weight_from_binary_counts(emo_pos_counts, total_samples)
    sen_weights = class_weights_from_counts(sen_counts, NUM_SENTIMENT)
    asp_weights = class_weights_from_counts(asp_counts, NUM_ASPECT)
    loss_fns = make_losses(emo_pos_weight, sen_weights, asp_weights)

    acc_scores = []
    for run in range(NUM_EPOCHS):
        model = MultiTaskWithEmojiAttention(pooling="mean").to(DEVICE)
        best_state, _, best_val_cache = train_model(
            model, train_loader, val_loader, DEVICE, NUM_EPOCHS, LR, loss_fns,   # <<< FIX: NUM_EPOCHS
            weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
            grad_accum_steps=GRAD_ACCUM, max_grad_norm=MAX_GRAD_NORM
        )

        # <<< FIX: safe multi-line unpack
        (val_logits_emotion, val_probs_emotion, val_true_emotion,
         val_logits_sen,     val_true_sen,     val_logits_asp,  val_true_asp) = best_val_cache

        emo_pred = (val_probs_emotion >= 0.5).astype(int)
        emo_sample_acc = np.mean((val_true_emotion == emo_pred).sum(axis=1) / val_true_emotion.shape[1])
        sent_acc = accuracy_score(val_true_sen, np.argmax(val_logits_sen, axis=1))
        aspect_acc = accuracy_score(val_true_asp, np.argmax(val_logits_asp, axis=1))
        acc_scores.append((emo_sample_acc, sent_acc, aspect_acc))

    acc_scores = np.array(acc_scores)
    emo_avg = float(np.mean(acc_scores[:, 0]))
    sent_avg = float(np.mean(acc_scores[:, 1]))
    asp_avg = float(np.mean(acc_scores[:, 2]))

    trial.set_user_attr("emotion_sample_accuracy", emo_avg)
    trial.set_user_attr("sentiment_accuracy", sent_avg)
    trial.set_user_attr("aspect_accuracy", asp_avg)

    final_score = (emo_avg + sent_avg + asp_avg) / 3.0
    trial.report(final_score, step=0)
    return final_score

# ========== RUN STUDY ==========
def run_optuna():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    # Save CSV
    rows = []
    for t in study.trials:
        row = {
            "trial": t.number,
            "value": t.value,
            "emotion_sample_accuracy": t.user_attrs.get("emotion_sample_accuracy", None),
            "sentiment_accuracy": t.user_attrs.get("sentiment_accuracy", None),
            "aspect_accuracy": t.user_attrs.get("aspect_accuracy", None),
        }
        for k, v in t.params.items():
            row[f"param_{k}"] = v
        rows.append(row)

    df_log = pd.DataFrame(rows)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    csv_path = os.path.join(CHECKPOINT_DIR, "optuna_trials_accuracy.csv")
    df_log.to_csv(csv_path, index=False)
    print(f"Saved trial log to: {csv_path}")

    # Plot **average** accuracy per trial
    valid_trials = df_log.dropna(subset=["emotion_sample_accuracy", "sentiment_accuracy","aspect_accuracy"]).sort_values("trial").reset_index(drop=True)
    avg_acc = valid_trials[["emotion_sample_accuracy", "sentiment_accuracy", "aspect_accuracy"]].mean(axis=1)

    x = np.arange(1, len(valid_trials) + 1)  # 1..N

    plt.figure(figsize=(10, 5))
    plt.plot(x, avg_acc, label="Average Accuracy", marker="o")
    plt.xlabel("Trial")
    plt.ylabel("Objective Accuracy")
    plt.title("Optuna Trials: Objective Accuracy vs Trial")
    plt.grid(True, alpha=0.3)
    plt.xlim(1, len(valid_trials))
    plt.xticks(x)
    plt.legend()

    plot_path = os.path.join(CHECKPOINT_DIR, "optuna_accuracy_plot.png")
    plt.savefig(plot_path, bbox_inches="tight", dpi=150)
    plt.close()
    print(f"Saved accuracy plot to: {plot_path}")

    print("\nBest trial:")
    for k, v in study.best_trial.params.items():
        print(f"{k}: {v}")
    print("Best Objective (avg accuracy):", study.best_value)

# ================== MAIN K-FOLD ==================
def main():
    df = pd.read_csv(TRAIN_CSV_PATH)
    dataset = MultiTaskDatasetWithEmoji(df, tokenizer, MAX_LEN)

    kfold = KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=42)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    fold_thresholds = []
    fold_temperatures = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f"\n========== Fold {fold+1}/{KFOLD_SPLITS} ==========")
        train_subset = Subset(dataset, train_idx)
        val_subset   = Subset(dataset, val_idx)
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False)

        emo_pos_counts, total_samples, sen_counts, asp_counts = collect_counts_from_subset(train_subset)
        emo_pos_weight = pos_weight_from_binary_counts(emo_pos_counts, total_samples)
        sen_weights = class_weights_from_counts(sen_counts, NUM_SENTIMENT)
        asp_weights = class_weights_from_counts(asp_counts, NUM_ASPECT)
        print("Emotion pos_weight:", emo_pos_weight.cpu().numpy().round(3).tolist())
        print("Sentiment weights :", sen_weights.cpu().numpy().round(3).tolist(), "counts:", sen_counts.tolist())
        print("Aspect weights    :", asp_weights.cpu().numpy().round(3).tolist(), "counts:", asp_counts.tolist())

        model = MultiTaskWithEmojiAttention(pooling="mean").to(DEVICE)
        loss_fns = make_losses(emo_pos_weight, sen_weights, asp_weights)

        best_state, best_loss, best_val_cache = train_model(
            model, train_loader, val_loader, DEVICE, NUM_EPOCHS, LR, loss_fns,   # <<< FIX: NUM_EPOCHS
            weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
            grad_accum_steps=GRAD_ACCUM, max_grad_norm=MAX_GRAD_NORM
        )
        model.load_state_dict(best_state)
        fold_model_path = os.path.join(CHECKPOINT_DIR, f"{BEST_NAME_MODEL}_{fold+1}.pt")
        torch.save(best_state, fold_model_path)
        print(f"Best model for Fold {fold+1} saved with Val Loss: {best_loss:.4f}")

        (val_logits_emotion, val_probs_emotion, val_true_emotion,
         _, _, _, _) = best_val_cache

        # Temperature scaling (opsional)
        T = tune_temperature_for_multilabel(val_true_emotion, val_logits_emotion)
        tuned_probs = 1.0 / (1.0 + np.exp(-(val_logits_emotion / T)))
        thresholds = tune_thresholds_per_label(val_true_emotion, tuned_probs)

        fold_temperatures.append(T)
        fold_thresholds.append(thresholds)
        print(f"Tuned temperature: {T:.3f}")
        print("Tuned thresholds (per label):", {lab: round(t, 3) for lab, t in zip(EMOTION_LABELS, thresholds)})

    # === Simpan ke DUA file JSON terpisah ===
    fold_thresholds = np.array(fold_thresholds)  # [K, C]
    avg_thresholds = fold_thresholds.mean(axis=0).tolist()
    avg_temperature = float(np.mean(fold_temperatures)) if len(fold_temperatures) > 0 else 1.0

    # pastikan direktori ada untuk keduanya
    os.makedirs(os.path.dirname(OUTPUT_THRESHOLDS), exist_ok=True)
    os.makedirs(os.path.dirname(OUTPUT_TEMPERATURE), exist_ok=True)

    # File 1: thresholds per label
    thresholds_payload = {lab: float(t) for lab, t in zip(EMOTION_LABELS, avg_thresholds)}
    with open(OUTPUT_THRESHOLDS, "w") as f:
        json.dump(thresholds_payload, f, indent=2)
    print(f"Saved averaged thresholds to {OUTPUT_THRESHOLDS}")

    # File 2: temperature saja
    temperature_payload = {"temperature": avg_temperature}
    with open(OUTPUT_TEMPERATURE, "w") as f:
        json.dump(temperature_payload, f, indent=2)
    print(f"Saved averaged temperature to {OUTPUT_TEMPERATURE}")

    # ============ INFERENCE (ENSEMBLE via LOGIT AVERAGE) ============
    print("\n[INFERENCE ENSEMBLE]")
    test_df = pd.read_csv(TEST_CSV_PATH)
    dataset_test = MultiTaskDatasetWithEmoji(test_df, tokenizer, MAX_LEN)
    test_loader = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

    all_logits_emotion = []; all_logits_sentiment = []; all_logits_aspect = []
    for fold in range(KFOLD_SPLITS):
        model = MultiTaskWithEmojiAttention(pooling="mean").to(DEVICE)
        model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, f"{BEST_NAME_MODEL}_{fold+1}.pt"), map_location=DEVICE))
        model.eval()
        fold_logits_emotion, fold_logits_sentiment, fold_logits_aspect = [], [], []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Inference Fold {fold+1}"):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                emoji_vec = batch['emoji_vec'].to(DEVICE)
                le, ls, la = model(input_ids, attention_mask, emoji_vec)
                fold_logits_emotion.append(le.cpu().numpy())
                fold_logits_sentiment.append(ls.cpu().numpy())
                fold_logits_aspect.append(la.cpu().numpy())
        all_logits_emotion.append(np.vstack(fold_logits_emotion))
        all_logits_sentiment.append(np.vstack(fold_logits_sentiment))
        all_logits_aspect.append(np.vstack(fold_logits_aspect))

    avg_logits_emotion   = np.mean(all_logits_emotion, axis=0)
    avg_logits_sentiment = np.mean(all_logits_sentiment, axis=0)
    avg_logits_aspect    = np.mean(all_logits_aspect, axis=0)

    # gunakan avg_temperature yang sudah dihitung (dan juga disimpan ke file terpisah)
    avg_probs_emotion   = 1.0 / (1.0 + np.exp(-(avg_logits_emotion / max(1e-9, avg_temperature))))
    avg_preds_sentiment = np.argmax(softmax_np(avg_logits_sentiment, axis=1), axis=1)
    avg_preds_aspect    = np.argmax(softmax_np(avg_logits_aspect, axis=1), axis=1)

    out_df = test_df.copy()
    for i, lab in enumerate(EMOTION_LABELS):
        out_df[f"proba_{lab}"] = avg_probs_emotion[:, i]
        out_df[f"pred_{lab}"]  = (avg_probs_emotion[:, i] >= avg_thresholds[i]).astype(int)
    out_df['pred_sentiment'] = avg_preds_sentiment
    out_df['pred_aspect']    = avg_preds_aspect

    os.makedirs(os.path.dirname(OUTPUT_PRED_BEST), exist_ok=True)
    out_df.to_csv(OUTPUT_PRED_BEST, index=False)
    print(f"Prediksi disimpan ke {OUTPUT_PRED_BEST}")

    # ====== METRICS EVALUATION (USING TUNED THRESHOLDS) ======
    print("\n[EVALUATION METRICS]")
    if set(EMOTION_LABELS).issubset(test_df.columns):
        y_true_emotion = test_df[EMOTION_LABELS].values.astype(int)
        emotion_metrics = evaluate_multilabel(y_true_emotion, avg_probs_emotion, thresholds_per_label=avg_thresholds)
        print_metrics(emotion_metrics, EMOTION_LABELS)
    else:
        print("Warning: Test CSV lacks ground-truth emotion columns; skipping emotion metrics.")

    if 'sentiment' in test_df.columns:
        y_true_sentiment = test_df['sentiment'].astype(int).values
        acc_sent = accuracy_score(y_true_sentiment, avg_preds_sentiment)
        prec_sent, rec_sent, f1_sent, _ = precision_recall_fscore_support(
            y_true_sentiment, avg_preds_sentiment, average='macro', zero_division=0
        )
        print("\n===== SENTIMENT (MULTI-CLASS) =====")
        print(f"Accuracy          : {acc_sent:.4f}")
        print(f"Precision         : {prec_sent:.4f}, Recall: {rec_sent:.4f}, F1: {f1_sent:.4f}")
        cw = precision_recall_fscore_support(y_true_sentiment, avg_preds_sentiment, average=None, zero_division=0)
        print("Per-class P/R/F1 :", {i: {"P": round(cw[0][i],4), "R": round(cw[1][i],4), "F1": round(cw[2][i],4)} for i in range(NUM_SENTIMENT)})
    else:
        print("Warning: Test CSV lacks 'sentiment'; skipping sentiment metrics.")

    if 'aspect' in test_df.columns:
        y_true_aspect = test_df['aspect'].astype(int).values
        acc_aspect = accuracy_score(y_true_aspect, avg_preds_aspect)
        prec_aspect, rec_aspect, f1_aspect, _ = precision_recall_fscore_support(
            y_true_aspect, avg_preds_aspect, average='macro', zero_division=0
        )
        print("\n===== ASPECT (MULTI-CLASS) =====")
        print(f"Accuracy          : {acc_aspect:.4f}")
        print(f"Precision         : {prec_aspect:.4f}, Recall: {rec_aspect:.4f}, F1: {f1_aspect:.4f}")
        cw = precision_recall_fscore_support(y_true_aspect, avg_preds_aspect, average=None, zero_division=0)
        print("Per-class P/R/F1 :", {i: {"P": round(cw[0][i],4), "R": round(cw[1][i],4), "F1": round(cw[2][i],4)} for i in range(NUM_ASPECT)})
    else:
        print("Warning: Test CSV lacks 'aspect'; skipping aspect metrics.")

if __name__ == "__main__":
    run_optuna()
